In [2]:
import torch
from torch import nn
import numpy as np
from sklearn import datasets
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split



In [3]:
bc = datasets.load_breast_cancer()
x, y = bc.data, bc.target

number_of_samples, number_of_features = x.shape
print(f"Number of samples: {number_of_samples}, Number of features: {number_of_features}")

Number of samples: 569, Number of features: 30


In [4]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

sc = StandardScaler()
x_train = sc.fit_transform(x_train)
x_test = sc.transform(x_test)

x_train = torch.from_numpy(x_train.astype(np.float32))
x_test = torch.from_numpy(x_test.astype(np.float32))
y_train = torch.from_numpy(y_train.astype(np.float32))
y_test = torch.from_numpy(y_test.astype(np.float32))

y_train = y_train.view(y_train.shape[0], 1)
y_test = y_test.view(y_test.shape[0],1)

In [5]:
class LogisticRegression(nn.Module):
    def __init__(self, n_input_features):
        super(LogisticRegression, self).__init__()
        self.linear = nn.Linear(n_input_features, 1)
        
    def forward(self, x):
        output = torch.sigmoid(self.linear(x))
        return output
    
model = LogisticRegression(number_of_features)
        

In [6]:
learning_rate = 0.01
criterion = nn.BCELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

num_epochs = 100

for epoch in range(num_epochs):
    outputs = model(x_train)
    loss = criterion(outputs, y_train)
    loss.backward()
    optimizer.step()
    
    optimizer.zero_grad()
    
    if (epoch +1) % 10 == 0:
        print(f"Epoch: {epoch+1}, Loss: {loss.item():.4f} ")
        
        
with torch.no_grad():
    outputs = model(x_test)
    predicted = outputs.round()
    accuracy = predicted.eq(y_test).sum() / float(y_test.shape[0])
    print(f"Accuracy: {accuracy:.4f}")
    

Epoch: 10, Loss: 0.4092 
Epoch: 20, Loss: 0.3663 
Epoch: 30, Loss: 0.3347 
Epoch: 40, Loss: 0.3102 
Epoch: 50, Loss: 0.2908 
Epoch: 60, Loss: 0.2748 
Epoch: 70, Loss: 0.2614 
Epoch: 80, Loss: 0.2500 
Epoch: 90, Loss: 0.2401 
Epoch: 100, Loss: 0.2314 
Accuracy: 0.9737
